# Anomaly Detection in Surveillance Videos using EfficientNet-B0

This notebook implements a **video anomaly detection** system using an **EfficientNet-B0** backbone.
Given a surveillance video as input, the model classifies the event occurring in it into one of **14 categories**:

| # | Class | # | Class |
|---|-------|---|-------|
| 0 | Abuse | 7 | Normal |
| 1 | Arrest | 8 | Road Accidents |
| 2 | Arson | 9 | Robbery |
| 3 | Assault | 10 | Shooting |
| 4 | Burglary | 11 | Shoplifting |
| 5 | Explosion | 12 | Stealing |
| 6 | Fighting | 13 | Vandalism |

**Why EfficientNet?**
- EfficientNet uses compound scaling (depth, width, resolution) for better accuracy with fewer parameters.
- EfficientNet-B0 has only ~5.3M params vs ResNet50's ~25.6M, making it faster to train.
- Better feature extraction due to squeeze-and-excitation blocks and inverted residuals.

**Approach:**
1. Uniformly sample N frames from the video.
2. Pass each frame through a pretrained EfficientNet-B0 feature extractor.
3. Temporally aggregate (average pool) frame-level features into a single video-level representation.
4. Classify using a fully connected head.

## 1. Install & Import Dependencies

In [ ]:
# Install required packages (uncomment if needed)
# !pip install torch torchvision opencv-python pandas scikit-learn matplotlib seaborn tqdm

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms
from sklearn.metrics import classification_report, confusion_matrix
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Check for GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Configuration

In [ ]:
# ======================== CONFIGURATION ========================
DATASET_ROOT = "dataset"              # Root folder containing train.csv, test.csv and class folders
TRAIN_CSV    = os.path.join(DATASET_ROOT, "train.csv")
TEST_CSV     = os.path.join(DATASET_ROOT, "test.csv")

IMG_SIZE          = 224               # EfficientNet-B0 default input size
FRAMES_PER_VIDEO  = 20               # Number of frames to uniformly sample per video
BATCH_SIZE        = 8                # EfficientNet is lighter, can use larger batches
EPOCHS            = 15
LEARNING_RATE     = 3e-4
WEIGHT_DECAY      = 1e-4             # L2 regularization
NUM_WORKERS       = 0                # Set > 0 if on Linux for faster data loading

# All 14 classes in the dataset
CLASSES = [
    'abuse', 'arrest', 'arson', 'assault', 'burglary',
    'explosion', 'fighting', 'normal', 'roadaccidents',
    'robbery', 'shooting', 'shoplifting', 'stealing', 'vandalism'
]
NUM_CLASSES  = len(CLASSES)
CLASS_TO_IDX = {label: idx for idx, label in enumerate(CLASSES)}
IDX_TO_CLASS = {idx: label for label, idx in CLASS_TO_IDX.items()}

print(f"Number of classes: {NUM_CLASSES}")
print(f"Classes: {CLASSES}")

## 3. Data Preparation

In [ ]:
def load_dataframe(csv_path):
    """Load CSV and fix video file paths."""
    df = pd.read_csv(csv_path)
    
    # Fix file paths:
    # CSV paths look like: data\normal\Normal_Videos_196_x264.mp4
    # Actual paths should be: dataset/normal/Normal_Videos_196_x264.mp4
    def fix_path(p):
        p = p.replace('\\', '/')          # Normalize separators
        if p.startswith('data/'):
            p = 'dataset/' + p[5:]         # Replace 'data/' prefix with 'dataset/'
        return p
    
    df['video_name'] = df['video_name'].apply(fix_path)
    df.reset_index(drop=True, inplace=True)
    
    return df

train_df = load_dataframe(TRAIN_CSV)
test_df  = load_dataframe(TEST_CSV)

print(f"Training samples: {len(train_df)}")
print(f"Testing samples:  {len(test_df)}")
print("\nSample entries:")
train_df.head()

### Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

train_counts = train_df['label'].value_counts().reindex(CLASSES, fill_value=0)
test_counts  = test_df['label'].value_counts().reindex(CLASSES, fill_value=0)

train_counts.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Training Set — Class Distribution')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

test_counts.plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Test Set — Class Distribution')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("Train distribution:\n", train_counts)
print("\nTest distribution:\n", test_counts)

### Compute Class Weights (to handle class imbalance)

In [ ]:
# ======================== CLASS IMBALANCE HANDLING ========================
# The 'normal' class is heavily over-represented. Without handling this,
# the model will learn to always predict 'normal' and plateau at ~50% accuracy.

# 1. Compute class weights for the loss function (inverse frequency)
label_counts = train_df['label'].value_counts()
total_samples = len(train_df)

class_weights = []
for cls in CLASSES:
    count = label_counts.get(cls, 1)  # avoid division by zero
    weight = total_samples / (NUM_CLASSES * count)
    class_weights.append(weight)

class_weights_tensor = torch.FloatTensor(class_weights).to(device)

print("Class weights for CrossEntropyLoss:")
for cls, w in zip(CLASSES, class_weights):
    count = label_counts.get(cls, 0)
    print(f"  {cls:15s}: weight={w:.3f}  (count={count})")

# 2. Compute per-sample weights for WeightedRandomSampler
#    This ensures each batch has a roughly balanced mix of classes
sample_weights = []
for _, row in train_df.iterrows():
    cls_idx = CLASS_TO_IDX[row['label']]
    sample_weights.append(class_weights[cls_idx])

sample_weights = torch.FloatTensor(sample_weights)
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
print(f"\nWeightedRandomSampler created with {len(sample_weights)} samples.")

### Verify Video File Existence

In [ ]:
# Quick check: verify that video files actually exist
missing_count = 0
for i, row in train_df.iterrows():
    if not os.path.exists(row['video_name']):
        missing_count += 1

if missing_count > 0:
    print(f"WARNING: {missing_count}/{len(train_df)} training video files not found!")
    print("Please ensure the video files are placed correctly under the 'dataset/' folder.")
else:
    print(f"All {len(train_df)} training video files found.")

## 4. Custom Video Dataset

In [ ]:
class VideoDataset(Dataset):
    """Custom Dataset for loading and sampling frames from videos."""
    
    def __init__(self, df, transform=None, frames_per_video=16):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.frames_per_video = frames_per_video

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        video_path = self.df.iloc[idx]['video_name']
        label_str  = self.df.iloc[idx]['label']
        label      = CLASS_TO_IDX[label_str]
        
        frames = self._load_video_frames(video_path)
        
        if self.transform:
            frames = torch.stack([self.transform(frame) for frame in frames])
        
        # Output shape: (frames_per_video, C, H, W)
        return frames, label

    def _load_video_frames(self, path):
        """Load video and uniformly sample frames."""
        cap = cv2.VideoCapture(path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        # Fallback for corrupt or missing videos
        if total_frames <= 0:
            cap.release()
            return [Image.new('RGB', (IMG_SIZE, IMG_SIZE)) for _ in range(self.frames_per_video)]
        
        # Uniformly sample frame indices across the entire video
        frame_indices = np.linspace(0, total_frames - 1, self.frames_per_video, dtype=int)
        
        frames = []
        for fidx in frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, fidx)
            ret, frame = cap.read()
            if ret:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames.append(Image.fromarray(frame))
            else:
                frames.append(frames[-1] if frames else Image.new('RGB', (IMG_SIZE, IMG_SIZE)))
        
        cap.release()
        
        while len(frames) < self.frames_per_video:
            frames.append(frames[-1] if frames else Image.new('RGB', (IMG_SIZE, IMG_SIZE)))
        
        return frames[:self.frames_per_video]

### Transforms & DataLoaders

In [ ]:
# ImageNet normalization values (used by pretrained EfficientNet)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),  # Resize slightly larger
    transforms.RandomCrop((IMG_SIZE, IMG_SIZE)),         # Then random crop to 224
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# Create datasets
train_dataset = VideoDataset(train_df, transform=train_transforms, frames_per_video=FRAMES_PER_VIDEO)
test_dataset  = VideoDataset(test_df,  transform=test_transforms,  frames_per_video=FRAMES_PER_VIDEO)

# Create data loaders — using WeightedRandomSampler for balanced batches
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,   num_workers=NUM_WORKERS)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches:  {len(test_loader)}")

### Sanity Check: Visualize Sample Frames

In [ ]:
# Visualize frames from a sample video
sample_frames, sample_label = train_dataset[0]
print(f"Sample tensor shape: {sample_frames.shape}")
print(f"Label: {IDX_TO_CLASS[sample_label]}")

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for i, ax in enumerate(axes.flat):
    if i < len(sample_frames):
        img = sample_frames[i].permute(1, 2, 0).numpy()
        img = img * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
        img = np.clip(img, 0, 1)
        ax.imshow(img)
        ax.set_title(f"Frame {i+1}")
    ax.axis('off')

plt.suptitle(f"Sample Video Frames — Class: {IDX_TO_CLASS[sample_label]}", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Model Architecture — EfficientNet-B0 Video Classifier

In [ ]:
class EfficientNetVideoClassifier(nn.Module):
    """
    Video classifier using EfficientNet-B0 as a frame-level feature extractor.
    
    Architecture:
        1. Each frame is passed through EfficientNet-B0 (pretrained on ImageNet)
           to extract a 1280-dim feature vector.
        2. Frame features are averaged (temporal pooling) to get one
           video-level feature vector.
        3. A fully connected classifier head maps the video feature
           to class logits.
    
    EfficientNet-B0 feature dim = 1280 (vs ResNet50's 2048)
    Total params ~5.3M (vs ResNet50's ~25.6M)
    """
    
    def __init__(self, num_classes, dropout_rate=0.3):
        super(EfficientNetVideoClassifier, self).__init__()
        
        # Load pretrained EfficientNet-B0
        efficientnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        
        # EfficientNet structure:
        # - efficientnet.features  -> convolutional feature extractor
        # - efficientnet.avgpool   -> adaptive average pooling
        # - efficientnet.classifier -> final FC layer
        
        # Keep features + avgpool as our feature extractor
        self.features = efficientnet.features
        self.avgpool  = efficientnet.avgpool
        
        # Selective fine-tuning:
        # EfficientNet-B0 has 9 feature blocks (0-8)
        # Freeze blocks 0-4 (early layers), fine-tune blocks 5-8 (later layers)
        for i, block in enumerate(self.features):
            if i < 5:
                for param in block.parameters():
                    param.requires_grad = False
            else:
                for param in block.parameters():
                    param.requires_grad = True
        
        # EfficientNet-B0 outputs 1280-dimensional features
        self.feature_dim = 1280
        
        # Classifier head with BatchNorm for stable training
        self.classifier = nn.Sequential(
            nn.Linear(self.feature_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        """
        Args:
            x: Tensor of shape (Batch, Frames, C, H, W)
        Returns:
            logits: Tensor of shape (Batch, num_classes)
        """
        batch_size, num_frames, c, h, w = x.shape
        
        # Reshape: merge batch and frame dims -> (B*F, C, H, W)
        x = x.view(batch_size * num_frames, c, h, w)
        
        # Extract frame-level features through EfficientNet
        features = self.features(x)       # -> (B*F, 1280, 7, 7)
        features = self.avgpool(features)  # -> (B*F, 1280, 1, 1)
        features = features.view(features.size(0), -1)  # -> (B*F, 1280)
        
        # Un-merge: -> (B, F, 1280)
        features = features.view(batch_size, num_frames, -1)
        
        # Temporal average pooling across frames -> (B, 1280)
        video_features = torch.mean(features, dim=1)
        
        # Classification
        logits = self.classifier(video_features)
        return logits

In [ ]:
# Initialize model
model = EfficientNetVideoClassifier(num_classes=NUM_CLASSES, dropout_rate=0.3)
model = model.to(device)

# Print model summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,} ({100*trainable_params/total_params:.1f}%)")
print(f"Frozen parameters:    {frozen_params:,} ({100*frozen_params/total_params:.1f}%)")

## 6. Training

In [ ]:
# ======================== LOSS, OPTIMIZER, SCHEDULER ========================

# Weighted CrossEntropyLoss to handle class imbalance
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

# Adam optimizer with weight decay (L2 regularization)
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Cosine Annealing scheduler — gradually reduces LR for better convergence
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

print(f"Loss:      CrossEntropyLoss (class-weighted)")
print(f"Optimizer: Adam (lr={LEARNING_RATE}, weight_decay={WEIGHT_DECAY})")
print(f"Scheduler: CosineAnnealingLR (T_max={EPOCHS})")

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    """Train the model for one epoch."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for videos, labels in tqdm(loader, desc="Training", leave=False):
        videos = videos.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(videos)
        loss = criterion(outputs, labels)
        loss.backward()
        
        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        running_loss += loss.item() * labels.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / total
    epoch_acc  = 100.0 * correct / total
    return epoch_loss, epoch_acc


def evaluate(model, loader, criterion):
    """Evaluate the model on a data loader."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for videos, labels in tqdm(loader, desc="Evaluating", leave=False):
            videos = videos.to(device)
            labels = labels.to(device)
            
            outputs = model(videos)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * labels.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / total
    epoch_acc  = 100.0 * correct / total
    return epoch_loss, epoch_acc

In [ ]:
# ======================== TRAINING LOOP ========================
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'lr': []}
best_val_acc = 0.0

for epoch in range(EPOCHS):
    current_lr = optimizer.param_groups[0]['lr']
    print(f"\nEpoch [{epoch+1}/{EPOCHS}]  (LR: {current_lr:.6f})")
    
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc     = evaluate(model, test_loader, criterion)
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)
    
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"  Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.2f}%")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_efficientnet_video_classifier.pth')
        print(f"  >> Best model saved (Val Acc: {val_acc:.2f}%)")

print(f"\nTraining complete. Best Validation Accuracy: {best_val_acc:.2f}%")

## 7. Training History Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

epochs_range = range(1, EPOCHS+1)

# Accuracy plot
axes[0].plot(epochs_range, history['train_acc'], 'b-o', label='Train Accuracy', markersize=4)
axes[0].plot(epochs_range, history['val_acc'],   'r-o', label='Val Accuracy', markersize=4)
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy (%)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss plot
axes[1].plot(epochs_range, history['train_loss'], 'b-o', label='Train Loss', markersize=4)
axes[1].plot(epochs_range, history['val_loss'],   'r-o', label='Val Loss', markersize=4)
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Learning rate plot
axes[2].plot(epochs_range, history['lr'], 'g-o', markersize=4)
axes[2].set_title('Learning Rate Schedule')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Learning Rate')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history_efficientnet.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Evaluation on Test Set

In [ ]:
# Load best model for evaluation
model.load_state_dict(torch.load('best_efficientnet_video_classifier.pth', map_location=device))
model.eval()

all_preds  = []
all_labels = []

with torch.no_grad():
    for videos, labels in tqdm(test_loader, desc="Testing"):
        videos = videos.to(device)
        outputs = model(videos)
        _, predicted = torch.max(outputs, 1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

# Classification Report
print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)
print(classification_report(all_labels, all_preds, target_names=CLASSES))

### Confusion Matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.title('Confusion Matrix — EfficientNet-B0')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('confusion_matrix_efficientnet.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Inference — Predict on a New Video

In [ ]:
def predict_video(video_path, model, transform=None):
    """
    Predict the class of a single video file.
    
    Args:
        video_path (str): Path to the video file.
        model: Trained EfficientNetVideoClassifier model.
        transform: Image transforms to apply.
    
    Returns:
        predicted_class (str): Predicted class name.
        confidence (float): Confidence score.
        probabilities (dict): Probability for each class.
    """
    if transform is None:
        transform = test_transforms
    
    model.eval()
    
    # Create a temporary dataframe for the VideoDataset
    temp_df = pd.DataFrame({'video_name': [video_path], 'label': ['normal']})
    temp_dataset = VideoDataset(temp_df, transform=transform, frames_per_video=FRAMES_PER_VIDEO)
    frames, _ = temp_dataset[0]
    
    # Add batch dimension: (1, Frames, C, H, W)
    frames = frames.unsqueeze(0).to(device)
    
    with torch.no_grad():
        outputs = model(frames)
        probs = torch.nn.functional.softmax(outputs, dim=1)
        confidence, predicted_idx = torch.max(probs, 1)
    
    predicted_class = IDX_TO_CLASS[predicted_idx.item()]
    confidence_val  = confidence.item()
    
    # Get probabilities for all classes
    probabilities = {IDX_TO_CLASS[i]: round(probs[0][i].item(), 4) for i in range(NUM_CLASSES)}
    
    print(f"Video: {video_path}")
    print(f"Prediction: {predicted_class.upper()} (Confidence: {confidence_val:.4f})")
    print(f"\nAll class probabilities:")
    for cls, prob in sorted(probabilities.items(), key=lambda x: x[1], reverse=True):
        bar = '█' * int(prob * 50)
        print(f"  {cls:15s} {prob:.4f} {bar}")
    
    return predicted_class, confidence_val, probabilities

In [ ]:
# ======================== EXAMPLE USAGE ========================
# Uncomment and replace with an actual video path to test:

# predicted_class, confidence, probs = predict_video(
#     "dataset/abuse/Abuse004_x264.mp4", model
# )